# 03 — Evaluation, Threshold Tuning & Grad-CAM
**Project:** Casting Defect Detection

**Goal of this notebook:** look deeper at the best model from `02_training.ipynb` (ResNet50V2):

1. **Confusion matrix**: exactly which mistakes does the model make?
2. **ROC and precision–recall curves**: how well does it separate OK from defective?
3. **Threshold tuning**: choose a decision threshold that misses as few defects as possible
4. **Error analysis**: look at the images the model got wrong
5. **Grad-CAM**: where does the model look when it decides a part is defective?

> Run on **Google Colab with a GPU** (Runtime → Change runtime type → T4 GPU → Run all).
> You need `resnet50v2.keras` from notebook 02 (see Section 2).

## 1. Setup

In [ ]:
import os
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
import keras
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score,
)

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT = 0.15          # must match notebook 02, so the validation set is the same images

FIG_DIR = "reports/figures"
MODEL_DIR = "models"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none")

## 2. Load the selected model
Notebook 02 saved every model under its own name (for example `resnet50v2.keras`) and selected
**ResNet50V2**. Set `MODEL_NAME` below if you want to evaluate a different model.

Choose **one** option to provide the model file:

- **Option A (recommended): Google Drive.** Put `resnet50v2.keras` in your Drive, for example in
  `MyDrive/casting-defect-detection/models/`.
- **Option B: Upload.** If the model is not found on Drive, you will be asked to upload
  `resnet50v2.keras` or the `training_outputs.zip` file from notebook 02.

In [ ]:
MODEL_NAME = "resnet50v2"   # the model selected in notebook 02
USE_DRIVE = True
DRIVE_MODEL_PATH = f"/content/drive/MyDrive/casting-defect-detection/models/{MODEL_NAME}.keras"
LOCAL_MODEL_PATH = f"{MODEL_DIR}/{MODEL_NAME}.keras"

def get_model_path():
    if os.path.exists(LOCAL_MODEL_PATH):
        return LOCAL_MODEL_PATH
    try:
        from google.colab import drive, files
    except ImportError:
        raise FileNotFoundError(f"Put {MODEL_NAME}.keras in {MODEL_DIR}/ and run again")

    if USE_DRIVE:
        drive.mount("/content/drive")
        if os.path.exists(DRIVE_MODEL_PATH):
            return DRIVE_MODEL_PATH
        print("Model not found on Drive:", DRIVE_MODEL_PATH)

    print(f"Upload {MODEL_NAME}.keras or training_outputs.zip")
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            with zipfile.ZipFile(name) as z:
                z.extractall(".")
        elif name.endswith(".keras"):
            os.replace(name, f"{MODEL_DIR}/{name}")
    if not os.path.exists(LOCAL_MODEL_PATH):
        raise FileNotFoundError(f"{MODEL_NAME}.keras was not found in the upload")
    return LOCAL_MODEL_PATH

MODEL_PATH = get_model_path()
model = keras.models.load_model(MODEL_PATH)
print("Loaded:", MODEL_PATH, "| model name:", model.name)

## 3. Load the validation and test sets

In [ ]:
!pip install -q kagglehub
import kagglehub

DATA_ROOT = kagglehub.dataset_download("ravirajsinh45/real-life-industrial-dataset-of-casting-product")

In [ ]:
def find_split_root(root):
    for dirpath, dirnames, _ in os.walk(root):
        if "train" in dirnames and "test" in dirnames:
            train_dir = os.path.join(dirpath, "train")
            if {"def_front", "ok_front"} <= set(os.listdir(train_dir)):
                return dirpath
    raise FileNotFoundError("Could not find train/test folders with def_front and ok_front")

SPLIT_ROOT = find_split_root(DATA_ROOT)
CLASS_NAMES = ["def_front", "ok_front"]

def load(directory, subset=None):
    return keras.utils.image_dataset_from_directory(
        directory, labels="inferred", label_mode="int", class_names=CLASS_NAMES,
        color_mode="rgb", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        shuffle=subset is not None, seed=SEED,
        validation_split=VAL_SPLIT if subset else None, subset=subset,
    )

# Same seed and split as notebook 02 -> the validation set is the same images the model never trained on
val_raw = load(os.path.join(SPLIT_ROOT, "train"), subset="validation")
test_raw = load(os.path.join(SPLIT_ROOT, "test"))
TEST_PATHS = test_raw.file_paths

def to_defect_label(images, labels):
    return images, tf.cast(1 - labels, tf.float32)   # defective = 1, ok = 0

val_ds = val_raw.map(to_defect_label).cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_raw.map(to_defect_label).cache().prefetch(tf.data.AUTOTUNE)

## 4. Predict
The model outputs a **probability of being defective** (0 to 1) for every image.

In [ ]:
def predict(ds):
    y_true = np.concatenate([y.numpy() for _, y in ds]).astype(int)
    y_prob = model.predict(ds, verbose=0).ravel()
    return y_true, y_prob

y_val, p_val = predict(val_ds)
y_test, p_test = predict(test_ds)
print(f"Validation: {len(y_val)} images | Test: {len(y_test)} images "
      f"({y_test.sum()} defective, {(1 - y_test).sum()} ok)")

## 5. Confusion matrix (threshold 0.5)
How to read it, for a quality inspector:

| | Predicted OK | Predicted defective |
|---|---|---|
| **Actually OK** | ✅ correct | ⚠️ **false alarm**: a good part is re-checked or scrapped |
| **Actually defective** | ❌ **missed defect**: a bad part reaches the customer | ✅ correct |

In [ ]:
LABELS = ["ok", "defective"]

def show_confusion(y_true, y_prob, threshold, title, fname):
    y_pred = (y_prob >= threshold).astype(int)
    cm_ = confusion_matrix(y_true, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(cm_, display_labels=LABELS)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{fname}", dpi=150)
    plt.show()
    tn, fp, fn, tp = cm_.ravel()
    return {"threshold": threshold, "missed defects (FN)": int(fn), "false alarms (FP)": int(fp),
            "recall": tp / (tp + fn), "precision": tp / (tp + fp) if (tp + fp) else 0.0,
            "accuracy": (tp + tn) / cm_.sum()}

result_default = show_confusion(y_test, p_test, 0.5, "Test set, threshold 0.5", "08_confusion_matrix.png")
print(classification_report(y_test, (p_test >= 0.5).astype(int), target_names=LABELS, digits=4))

## 6. ROC and precision–recall curves
- **ROC curve:** the closer the curve hugs the top-left corner, the better the model separates the classes
  (AUC = 1.0 is perfect).
- **Precision–recall curve:** shows the trade-off between catching every defect (recall) and avoiding
  false alarms (precision).

In [ ]:
fpr, tpr, _ = roc_curve(y_test, p_test)
prec, rec, _ = precision_recall_curve(y_test, p_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(fpr, tpr, label=f"AUC = {auc(fpr, tpr):.4f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(title="ROC curve (test)", xlabel="False positive rate", ylabel="True positive rate (recall)")
axes[0].legend(loc="lower right")

axes[1].plot(rec, prec, label=f"AP = {average_precision_score(y_test, p_test):.4f}")
axes[1].set(title="Precision–recall curve (test)", xlabel="Recall (defective)", ylabel="Precision")
axes[1].legend(loc="lower left")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/09_roc_pr_curves.png", dpi=150)
plt.show()

## 7. Threshold tuning
By default a part is called defective if its probability is ≥ 0.5. **Lowering** the threshold catches
more defects but raises more false alarms.

The threshold is chosen on the **validation set**, never on the test set: picking it on the test set
would be "peeking at the answers" and make the final score look better than it really is.

**Rule:** pick the **highest threshold that still reaches the target recall** on validation. The highest
one keeps false alarms as low as possible while meeting the recall target.

In [ ]:
TARGET_RECALL = 1.0   # goal: no missed defects on the validation set (lower to e.g. 0.995 if needed)

thresholds = np.round(np.arange(0.01, 0.991, 0.01), 2)
rows = []
for t in thresholds:
    pred = (p_val >= t).astype(int)
    tp = int(((pred == 1) & (y_val == 1)).sum())
    fn = int(((pred == 0) & (y_val == 1)).sum())
    fp = int(((pred == 1) & (y_val == 0)).sum())
    rows.append({"threshold": t, "recall": tp / (tp + fn), "precision": tp / (tp + fp) if (tp + fp) else 0.0,
                 "missed": fn, "false_alarms": fp})
sweep = pd.DataFrame(rows)

meets_target = sweep[sweep["recall"] >= TARGET_RECALL]
if len(meets_target):
    BEST_THRESHOLD = float(meets_target["threshold"].max())
else:
    BEST_THRESHOLD = float(sweep.loc[sweep["recall"].idxmax(), "threshold"])
    print(f"Target recall {TARGET_RECALL} not reached; using the threshold with the highest recall")
print("Chosen threshold (from validation):", BEST_THRESHOLD)

plt.figure(figsize=(8, 4))
plt.plot(sweep["threshold"], sweep["recall"], label="recall (defective)")
plt.plot(sweep["threshold"], sweep["precision"], label="precision")
plt.axvline(0.5, color="gray", linestyle=":", label="default 0.5")
plt.axvline(BEST_THRESHOLD, color="red", linestyle="--", label=f"chosen {BEST_THRESHOLD:.2f}")
plt.ylim(min(sweep[["recall", "precision"]].min().min(), 0.9) - 0.01, 1.005)
plt.xlabel("Decision threshold")
plt.title("Recall and precision vs. threshold (validation set)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/10_threshold_tuning.png", dpi=150)
plt.show()

In [ ]:
# Apply the chosen threshold to the untouched test set and compare with the default 0.5
result_tuned = show_confusion(y_test, p_test, BEST_THRESHOLD,
                              f"Test set, threshold {BEST_THRESHOLD:.2f}", "08b_confusion_matrix_tuned.png")
threshold_comparison = pd.DataFrame([result_default, result_tuned]).set_index("threshold").round(4)
threshold_comparison.to_csv("reports/threshold_comparison.csv")
threshold_comparison

In [ ]:
# Save the threshold so the demo app uses the same decision rule
with open(f"{MODEL_DIR}/threshold.json", "w") as f:
    json.dump({"model": MODEL_NAME, "threshold": BEST_THRESHOLD, "chosen_on": "validation",
               "target_recall": TARGET_RECALL}, f, indent=2)
print(open(f"{MODEL_DIR}/threshold.json").read())

## 8. Error analysis: which images did the model get wrong?
Looking at the mistakes tells us whether they are real model weaknesses or borderline / possibly
mislabeled parts.

In [ ]:
def show_images(indices, title, fname, max_n=8):
    indices = list(indices)[:max_n]
    if not indices:
        print(f"{title}: none 🎉")
        return
    fig, axes = plt.subplots(1, len(indices), figsize=(2.6 * len(indices), 3), squeeze=False)
    for ax, i in zip(axes[0], indices):
        img = keras.utils.load_img(TEST_PATHS[i], target_size=IMG_SIZE)
        ax.imshow(img)
        ax.set_title(f"true: {LABELS[y_test[i]]}\np(defect)={p_test[i]:.3f}", fontsize=9)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{fname}", dpi=150)
    plt.show()

pred_tuned = (p_test >= BEST_THRESHOLD).astype(int)
missed = np.where((y_test == 1) & (pred_tuned == 0))[0]
false_alarms = np.where((y_test == 0) & (pred_tuned == 1))[0]
print(f"Missed defects: {len(missed)} | False alarms: {len(false_alarms)}  (threshold {BEST_THRESHOLD:.2f})")

show_images(missed, "Missed defects (defective parts predicted as OK)", "12_missed_defects.png")
show_images(false_alarms, "False alarms (OK parts predicted as defective)", "12_false_alarms.png")

## 9. Grad-CAM: where does the model look?
**Grad-CAM** [Selvaraju et al., 2017] highlights the image regions that pushed the model toward its
decision. Red = strong influence, blue = little influence.

From the EDA we expect the model to focus on the **outer and inner rims**, where most defects were visible.
If the heatmaps agree, the model has learned meaningful features rather than shortcuts
(for example, background or lighting).

How it works here: the model is split into (1) everything up to the backbone's last feature map and
(2) the classification head. The gradients of the "defective" score with respect to that feature map
tell us how important each spatial location is.

In [ ]:
def split_model(model):
    """Return (pre-layers, backbone, head-layers) for a transfer-learning model from notebook 02."""
    layer_list = [l for l in model.layers if not isinstance(l, keras.layers.InputLayer)]
    backbone_idx = next((i for i, l in enumerate(layer_list)
                         if isinstance(l, keras.Model) and l.name != "augmentation"), None)
    if backbone_idx is None:
        raise ValueError("Grad-CAM here expects a transfer-learning model with a pretrained backbone")
    pre = [l for l in layer_list[:backbone_idx] if l.name != "augmentation"]   # no augmentation at inference
    return pre, layer_list[backbone_idx], layer_list[backbone_idx + 1:]

PRE_LAYERS, BACKBONE, HEAD_LAYERS = split_model(model)
print("Backbone:", BACKBONE.name, "| pre:", [l.name for l in PRE_LAYERS], "| head:", [l.name for l in HEAD_LAYERS])

def grad_cam(img_array):
    """img_array: (H, W, 3) raw image with values 0-255. Returns (heatmap 0-1, p_defect)."""
    x = tf.convert_to_tensor(img_array[None, ...], dtype=tf.float32)
    for layer in PRE_LAYERS:
        x = layer(x)
    with tf.GradientTape() as tape:
        features = BACKBONE(x, training=False)
        tape.watch(features)
        y = features
        for layer in HEAD_LAYERS[:-1]:
            y = layer(y, training=False)
        # Use the LOGIT (score before the sigmoid). On confident predictions (p close to 1)
        # the sigmoid saturates and its gradient vanishes, which would give an empty heatmap.
        dense = HEAD_LAYERS[-1]
        logit = tf.matmul(y, dense.kernel) + dense.bias
        score = logit[:, 0]
    grads = tape.gradient(score, features)[0]              # (h, w, channels)
    weights = tf.reduce_mean(grads, axis=(0, 1))           # importance of each channel
    cam = tf.nn.relu(tf.reduce_sum(features[0] * weights, axis=-1))
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return cam.numpy(), float(tf.sigmoid(score[0]))

def overlay(img_array, heatmap, alpha=0.4):
    heat = tf.image.resize(heatmap[..., None], IMG_SIZE).numpy()[..., 0]
    colored = cm.jet(heat)[..., :3] * 255
    return np.clip((1 - alpha) * img_array + alpha * colored, 0, 255).astype("uint8")

In [ ]:
def show_gradcam(indices, title, fname, max_n=6):
    indices = list(indices)[:max_n]
    if not indices:
        print(f"{title}: no images")
        return
    fig, axes = plt.subplots(2, len(indices), figsize=(2.8 * len(indices), 6), squeeze=False)
    for col, i in enumerate(indices):
        img = keras.utils.img_to_array(keras.utils.load_img(TEST_PATHS[i], target_size=IMG_SIZE))
        heatmap, p = grad_cam(img)
        axes[0, col].imshow(img.astype("uint8"))
        axes[0, col].set_title(f"true: {LABELS[y_test[i]]}\np(defect)={p:.3f}", fontsize=9)
        axes[1, col].imshow(overlay(img, heatmap))
        axes[1, col].set_title("Grad-CAM", fontsize=9)
        for ax in axes[:, col]:
            ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{fname}", dpi=150)
    plt.show()

rng = np.random.default_rng(SEED)
correct_defects = np.where((y_test == 1) & (pred_tuned == 1))[0]
correct_ok = np.where((y_test == 0) & (pred_tuned == 0))[0]

show_gradcam(rng.choice(correct_defects, size=min(6, len(correct_defects)), replace=False),
             "Grad-CAM: correctly detected defects", "11_gradcam_defective.png")
show_gradcam(rng.choice(correct_ok, size=min(6, len(correct_ok)), replace=False),
             "Grad-CAM: correctly classified OK parts", "11_gradcam_ok.png")
show_gradcam(np.concatenate([missed, false_alarms]).astype(int),
             "Grad-CAM: model mistakes", "11_gradcam_mistakes.png")

**How to read the heatmaps:**
- On **defective** parts, red areas should sit on the visible damage, often at the outer or inner rim.
- On **OK** parts, the "defective" heatmap is usually weak or spread out, because there is nothing defective to point at.
- If red areas often sit on the **background** instead of the part, the model may be using a shortcut, which is
  a warning sign even when the scores are high.

The backbone's last feature map is only 7×7, so the heatmaps are coarse: they show the **region**, not the exact pixel.

## 10. Download the outputs

In [ ]:
!zip -qr evaluation_outputs.zip reports models/threshold.json
try:
    from google.colab import files
    files.download("evaluation_outputs.zip")
except ImportError:
    print("Not running on Colab: outputs are in the reports/ and models/ folders")

## 11. Summary
*(Fill this in after running. This goes into the README.)*

- Confusion matrix at threshold 0.5: … missed defects, … false alarms
- Chosen threshold (on validation): … → on the test set: … missed defects, … false alarms
- Error analysis: what do the misclassified images look like? (borderline, unclear, possibly mislabeled?)
- Grad-CAM: does the model focus on the outer/inner rims and visible damage, as seen in the EDA?

**Next:** Streamlit demo app (`app/streamlit_app.py`) using `resnet50v2.keras` and `threshold.json`.